In [53]:
import numpy as np
import pandas as pd
from statsmodels.duration.hazard_regression import PHReg

In [54]:
np.random.seed(16)

In [55]:
def fp_transform(t, p1, p2):
    t = np.clip(t, 1e-6, None) # Prevent log(0)
    term1 = np.power(t, p1)
    if np.isclose(p1, p2):
        term2 = term1 * np.log(t)
    else:
        term2 = np.power(t, p2)
    return term1, term2

In [56]:
def fitness(powers, df, covariate, time_col, event_col='Event'):
    # Small epsilon to avoid log(0) or division by zero with negative powers
    t = df[time_col].values + 1e-6 
    X = df[covariate].values
    
    try:
        if len(powers) == 1:
            fp_term = fp_transform(t, powers[0])
        else:
            fp_term = fp_transform(t, powers[0], powers[1])
        
        exog = pd.DataFrame({'X': X, 'X_fp': X * fp_term})
        endog = df[[time_col, event_col]]
        
        model = PHReg(endog=endog, exog=exog)
        res = model.fit()
        
        # Ensure the log-likelihood is a valid number
        if not np.isfinite(res.llf):
            return -1e10
            
        aic = -2 * res.llf + 2 * (len(powers) + 1)
        return -aic
    except:
        # Catch convergence errors, singular matrices, or overflows
        return -1e10

In [57]:
def genetic_algorithm(df, covariate, time, fp_degree=2, pop_size=50, generations=100, mutation_rate=0.1):
    bounds = [-2, 3]
    pop = np.random.uniform(bounds[0], bounds[1], (pop_size, fp_degree))
    
    # Initialize bests with the first individual to avoid None errors
    best_overall_params = pop[0].copy()
    best_overall_fit = -np.inf

    for gen in range(generations):
        # 1. Evaluate Fitness and replace any remaining NaNs with a very low number
        fits = np.array([fitness(ind, df, covariate, time) for ind in pop])
        fits = np.nan_to_num(fits, nan=-1e10, neginf=-1e10)
        
        # 2. Track the best solution
        current_best_idx = np.argmax(fits) # Use np.argmax since we cleaned NaNs
        if fits[current_best_idx] > best_overall_fit:
            best_overall_fit = fits[current_best_idx]
            best_overall_params = pop[current_best_idx].copy()

        # 3. Selection (Tournament)
        new_pop = []
        # Elitism: keep the best individual
        new_pop.append(best_overall_params)
        
        while len(new_pop) < pop_size:
            # Tournament selection: Pick 3 random indices
            idx = np.random.randint(0, pop_size, 3)
            
            # Robust selection: if all 3 are equally bad, just pick the first
            tournament_fits = fits[idx]
            winner_idx = idx[np.argmax(tournament_fits)]
            parent1 = pop[winner_idx]
            
            # Repeat for parent 2
            idx = np.random.randint(0, pop_size, 3)
            parent2 = pop[idx[np.argmax(fits[idx])]]
            
            # 4. Crossover (Arithmetic)
            alpha = np.random.random()
            child = alpha * parent1 + (1 - alpha) * parent2
            
            # 5. Mutation
            if np.random.random() < mutation_rate:
                child += np.random.normal(0, 0.1, size=fp_degree)
            
            new_pop.append(np.clip(child, bounds[0], bounds[1]))
        
        pop = np.array(new_pop)
        
        if gen % 10 == 0:
            print(f"Gen {gen} | Best AIC: {-best_overall_fit:.2f}")

    return best_overall_params, best_overall_fit

In [58]:
df = pd.read_csv("../data/preprocess-data/preprocess_simulated_data.csv")
time = "Time"
covariate = "Age"

In [59]:
best_overall_params, best_overall_fit = genetic_algorithm(df, "Age", "Time")
print(f"Best Powers: {best_overall_params} & Best Fit: {best_overall_fit}")

Gen 0 | Best AIC: 10000000000.00
Gen 10 | Best AIC: 10000000000.00
Gen 20 | Best AIC: 10000000000.00
Gen 30 | Best AIC: 10000000000.00
Gen 40 | Best AIC: 10000000000.00
Gen 50 | Best AIC: 10000000000.00
Gen 60 | Best AIC: 10000000000.00
Gen 70 | Best AIC: 10000000000.00
Gen 80 | Best AIC: 10000000000.00
Gen 90 | Best AIC: 10000000000.00
Best Powers: [-0.8835446   0.61581671] & Best Fit: -10000000000.0
